# Fig. 5d-f: GSDensity base scores of the Xenium breast cancer section

Per-cell GSDensity relevance (MCA co-embedding, 300-NN graph, random walk with restart; nn_use 300, restart 0.75, MCA
dimensions 1-10, seed 0) of the DCIS and CD8+ T cell signatures of 03_xenium_platform and their held-out versions
(R4_F2f_downstream_platforms), on all 139,320 cells, run in the R environment gsdensity_env. A companion run on a
spatially stratified subsample of 5,000 cells (few ties at relevance 0) is written next to it. One CSV per signature
goes to data/gsd_cache/xenium/ (and xenium/stratified_5000/) for the figure notebooks.

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

In [3]:
sc.set_figure_params(frameon=False, fontsize=8)
sc.settings._vector_friendly = False
plt.rcParams["axes.grid"] = False

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
os.makedirs(project_dir + "logs", exist_ok=True)
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"
os.makedirs(source_dir, exist_ok=True)


# GSDensity scores of this dataset (one CSV per signature) and the R task directories
gsd_dir = working_dir + "data/gsd_cache/xenium/"
task_root = working_dir + "data/rev_cache/GSD_base_xenium/"

In [5]:
from helpers import (
    export_gsdensity_task,
    run_gsdensity,
    read_gsdensity_outputs,
    write_gsdensity_cache,
    spatial_grid_subsample,
)

In [6]:
# 1. Load data (03_xenium_platform cells 5-6, 10)
adata = sc.read_h5ad(project_dir + "xenium_aligned_ductal_invasive.h5ad")
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f"{adata.n_obs} cells x {adata.n_vars} genes")

139320 cells x 313 genes


In [7]:
# 2. Gene sets: DCIS and CD8+ T cells (03_xenium_platform cells 7-9, top-10 pseudo-bulk DESeq2 genes) and the
# held-out sets of R4_F2f_downstream_platforms (DEGs from cells left of the x-median)
signatures_dict = {
    "DCIS": ["CEACAM6", "AGR3", "CEACAM8", "SCGB2A1", "ESR1", "SERPINA3", "TACSTD2", "CLDN4", "HPX", "S100A14"],
    "CD8+_T_Cells": ["KLRD1", "CD8A", "GZMA", "KLRC1", "NKG7", "GNLY", "CCL5", "PRF1", "CD8B", "CD3G"],
    "DCIS_heldout_left": ["AGR3", "CEACAM6", "SERPINA3", "CEACAM8", "ESR1", "SCGB2A1", "TACSTD2", "CLDN4", "S100A14", "LYPD3"],
    "CD8+_T_Cells_heldout_left": ["KLRD1", "GNLY", "CD8A", "NKG7", "GZMA", "KLRC1", "CCL5", "CD8B", "PRF1", "CD3G"],
}
for s, genes in signatures_dict.items():
    print(f"{s}: {', '.join(genes)}")

DCIS: CEACAM6, AGR3, CEACAM8, SCGB2A1, ESR1, SERPINA3, TACSTD2, CLDN4, HPX, S100A14
CD8+_T_Cells: KLRD1, CD8A, GZMA, KLRC1, NKG7, GNLY, CCL5, PRF1, CD8B, CD3G
DCIS_heldout_left: AGR3, CEACAM6, SERPINA3, CEACAM8, ESR1, SCGB2A1, TACSTD2, CLDN4, S100A14, LYPD3
CD8+_T_Cells_heldout_left: KLRD1, GNLY, CD8A, NKG7, GZMA, KLRC1, CCL5, CD8B, PRF1, CD3G


In [8]:
# 3. GSDensity bridge: R package in the conda environment gsdensity_env, run by scripts/run_gsdensity.R
gsd_rscript = working_dir + "scripts/run_gsdensity.R"
gsd_params = {"nn_use": 300, "restart": 0.75, "dims": 10, "seed": 0}
index_params = dict(gsd_params, spatial=True, spatial_n=10, spatial_ntimes=100)  # spatial KLD at the R defaults

# GSDensity passes the whole gene-cell graph (about 916 MiB here) through future.apply, above future's 500 MiB size
# guard; the guard (a size check only) is raised for the R session through this environment variable
os.environ["R_FUTURE_GLOBALS_MAXSIZE"] = str(20 * 1024**3)

In [9]:
# 4. Full section: one task (all gene sets share one MCA embedding and graph), run in one R session
task_full = task_root + "full/"
export_gsdensity_task(adata, signatures_dict, task_full, gsd_params)
t0 = time.time()
rc = run_gsdensity([task_full], gsd_rscript, n_jobs=1, log=working_dir + "logs/REV_GSD_base_xenium_full.log")
print(f"R exit code: {rc}; wall clock: {time.time() - t0:.0f} s")

R exit code: [0]; wall clock: 851 s


`dnet::dRWR` (inside GSDensity's random walk) sets steady-state probabilities below 1e-6 to zero. On a graph of
about 100,000 spots or cells plus genes, most of them therefore tie at relevance 0; this is native GSDensity behaviour
and is counted in the index (`n_zero_relevance`).

In [10]:
# 5. One CSV per signature (with the Xenium cell_id) and the index
gsd_full = read_gsdensity_outputs(task_full)
index = write_gsdensity_cache(
    adata,
    gsd_full,
    signatures_dict,
    gsd_dir,
    working_dir + "data/gsd_cache/INDEX_xenium.csv",
    dataset="xenium",
    params=index_params,
    note=f"none: full data (all {adata.n_obs:,} cells), one MCA embedding",
    n_total=adata.n_obs,
    working_dir=working_dir,
    extra_columns={"cell_id": adata.obs["cell_id"].astype(str).values},
)
print(index[["signature", "genes_used_of_given", "n_spots", "n_positive_label", "n_zero_relevance", "runtime_s"]]
      .to_string(index=False))

                signature genes_used_of_given  n_spots  n_positive_label  n_zero_relevance  runtime_s
                     DCIS               10/10   139320                32            129012      832.9
             CD8+_T_Cells               10/10   139320                 0            128327      832.9
        DCIS_heldout_left               10/10   139320                 0            129139      832.9
CD8+_T_Cells_heldout_left               10/10   139320                 0            128327      832.9


In [11]:
# 6. Companion run: spatially stratified subsample of 5,000 cells (50 x 50 grid, seed 0), its own MCA embedding
keep = spatial_grid_subsample(np.asarray(adata.obsm["spatial"], float)[:, :2], 5000, n_bins=50, seed=0)
sub = adata[keep].copy()
task_sub = task_root + "test_5000/"
export_gsdensity_task(sub, signatures_dict, task_sub, gsd_params)
t0 = time.time()
rc = run_gsdensity([task_sub], gsd_rscript, n_jobs=1, log=working_dir + "logs/REV_GSD_base_xenium_stratified_5000.log")
print(f"R exit code: {rc}; wall clock: {time.time() - t0:.0f} s")

R exit code: [0]; wall clock: 42 s


In [12]:
# 7. Companion CSVs and index in xenium/stratified_5000/ (compare methods on the same cells, spot_id)
companion_dir = gsd_dir + "stratified_5000/"
gsd_sub = read_gsdensity_outputs(task_sub)
index_sub = write_gsdensity_cache(
    sub,
    gsd_sub,
    signatures_dict,
    companion_dir,
    companion_dir + "INDEX.csv",
    dataset="xenium",
    params=index_params,
    note=(f"companion run (not the base cache): spatially stratified subsample of {sub.n_obs:,} of {adata.n_obs:,} "
          "cells (50 x 50 grid over the section, cells drawn per occupied bin in proportion to its size, seed 0), one "
          "MCA embedding of the subsample"),
    n_total=adata.n_obs,
    working_dir=working_dir,
    extra_columns={"cell_id": sub.obs["cell_id"].astype(str).values},
)
print(index_sub[["signature", "n_spots", "n_positive_label", "n_zero_relevance", "runtime_s"]].to_string(index=False))

                signature  n_spots  n_positive_label  n_zero_relevance  runtime_s
                     DCIS     5000               789               172       23.3
             CD8+_T_Cells     5000               359               774       23.3
        DCIS_heldout_left     5000               846               155       23.3
CD8+_T_Cells_heldout_left     5000               359               774       23.3
